# Project 5: Working with Pandas and SQL Databases (Movies Dataset)

# Project Brief for Self-Coders

Here you´ll have the opportunity to code major parts of Project 5 on your own. If you need any help or inspiration, have a look at the Videos or the Jupyter Notebook with the full code. <br> <br>
Keep in mind that it´s all about __getting the right results/conclusions__. It´s not about finding the identical code. Things can be coded in many different ways. Even if you come to the same conclusions, it´s very unlikely that we have the very same code. 

## Creating an SQLite Database

1. __Import__ sqlite3 (as sq3) and __create__ a new SQLite Database with the name __"movies.db"__.

In [309]:
import sqlite3 as sq3
import pandas as pd
import numpy as np


In [310]:
database = sq3.connect('movies.db')

## Loading Data from DataFrames into an SQLite Database

2. __Load__ the json file __"some_movies.json"__ and __split__ the dataset into the following __four datasets__ (save each dataset in a Pandas DataFrame).

In [311]:
some_movies = pd.read_json('some_movies.json')


In [312]:
some_movies.columns

Index(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres',
       'homepage', 'id', 'imdb_id', 'original_language', 'original_title',
       'overview', 'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

__Dataset #1 (Movies)__ with columns ["id", "title", "revenue", "budget", "belongs_to_collection_name", "release_date"]. <br>
Convert "release_date" to datetime and transform "budget" and "revenue" to Million USD before loading into the Database. 

In [313]:
columns_to_be_numereic = ['budget','revenue']
some_movies['budget'] = pd.to_numeric(some_movies['budget'],errors='coerce')
some_movies['revenue'] = pd.to_numeric(some_movies['revenue'],errors='coerce')
some_movies['release_date'] = pd.to_datetime(some_movies['release_date'],errors='coerce')
some_movies['budget'] = some_movies['budget']/1000000
some_movies['revenue'] = some_movies['revenue']/1000000


In [314]:
some_movies['belongs_to_collection'][0]

{'id': 86311,
 'name': 'The Avengers Collection',
 'poster_path': '/yFSIUVTCvgYrpalUktulvk3Gi5Y.jpg',
 'backdrop_path': '/zuW6fOiusv4X9nnW3paHGfXcSll.jpg'}

In [315]:
some_movies['belongs_to_collection'] = some_movies['belongs_to_collection'].apply(lambda x: x['name'] if isinstance(x, dict) and 'name' in x else '')

In [316]:
some_movies['belongs_to_collection'][0]

'The Avengers Collection'

In [317]:
movies = some_movies[["id", "title", "revenue", "budget", "belongs_to_collection", "release_date"]]

In [318]:
movies.value_counts().sum()

18

In [319]:
movies.head()

,id,title,revenue,budget,belongs_to_collection,release_date
0,299534,Avengers: Endgame,2797.800564,356.0,The Avengers Collection,2019-04-24
1,19995,Avatar,2787.965087,237.0,Avatar Collection,2009-12-10
2,140607,Star Wars: The Force Awakens,2068.223624,245.0,Star Wars Collection,2015-12-15
3,299536,Avengers: Infinity War,2046.239637,300.0,The Avengers Collection,2018-04-25
4,597,Titanic,1845.034188,200.0,,1997-11-18


__Dataset #2 (Votes)__ with columns ["id", "vote_count", "vote_average"]. 

In [320]:
votes = some_movies[['id','vote_count','vote_average']]

In [321]:
votes.head()

,id,vote_count,vote_average
0,299534,12607,8.3
1,19995,21000,7.4
2,140607,14205,7.4
3,299536,17718,8.3
4,597,16661,7.8


__Dataset #3 (Genres)__ with columns ["genre_id", "genre_name", "id"]. <br> 

In [322]:
some_movies['genres'][0]


[{'id': 12, 'name': 'Adventure'},
 {'id': 878, 'name': 'Science Fiction'},
 {'id': 28, 'name': 'Action'}]

In [323]:
some_movies['genre_id'] = some_movies['genres'].apply(lambda x: ([d['id'] for d in x]) if isinstance(x,list) else np.nan)
some_movies['genre_name'] = some_movies['genres'].apply(lambda x : '|'.join([d['name'] for d in x]) if isinstance(x,list) else np.nan)

In [324]:
genres = some_movies[['genre_id','genre_name','id']]

In [325]:
genres['genre_id'] = genres['genre_id'].apply(lambda x: ','.join(map(str, x)) if isinstance(x, list) else '')
genres.head()

C:\Users\ahmed\AppData\Local\Temp\ipykernel_13952\3314421399.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  genres['genre_id'] = genres['genre_id'].apply(lambda x: ','.join(map(str, x)) if isinstance(x, list) else '')


,genre_id,genre_name,id
0,"12,878,28",Adventure|Science Fiction|Action,299534
1,"28,12,14,878",Action|Adventure|Fantasy|Science Fiction,19995
2,"28,12,878,14",Action|Adventure|Science Fiction|Fantasy,140607
3,"12,28,878",Adventure|Action|Science Fiction,299536
4,"18,10749",Drama|Romance,597


__Dataset #4 (Prod)__ with columns ["comp_id", "comp_logo_path", "comp_name", "comp_origin_country", "id" ]. <br>


In [326]:
some_movies['production_companies'][0]

[{'id': 420,
  'logo_path': '/hUzeosd33nzE5MCNsZxCGEKTXaQ.png',
  'name': 'Marvel Studios',
  'origin_country': 'US'}]

In [327]:
some_movies['comp_id'] = some_movies['production_companies'].apply(lambda x: [d['id'] for d in x] if isinstance(x,list) else np.nan)
some_movies['comp_name'] = some_movies['production_companies'].apply(lambda x : '|'.join([d['name'] for d in x]) if isinstance(x,list) else np.nan)
some_movies['comp_origin_country'] = some_movies['production_companies'].apply(lambda x:'|'.join( [d['origin_country'] for d in x]) if isinstance(x,list) else np.nan)
some_movies['comp_logo_path'] = some_movies['production_companies'].apply(
    lambda x: x[0]['logo_path'] if isinstance(x, list) and len(x) > 0 and 'logo_path' in x[0] else np.nan)

In [328]:
prod = some_movies[["comp_id", "comp_logo_path", "comp_name", "comp_origin_country", "id"]]
#prod['comp_logo_path'] = prod["comp_logo_path"].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
prod['comp_id'] = prod['comp_id'].apply(lambda x : ','.join(map(str,x)) if isinstance(x,list) else '')


C:\Users\ahmed\AppData\Local\Temp\ipykernel_13952\2312888765.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prod['comp_id'] = prod['comp_id'].apply(lambda x : ','.join(map(str,x)) if isinstance(x,list) else '')


In [329]:
prod.head()

,comp_id,comp_logo_path,comp_name,comp_origin_country,id
0,420,/hUzeosd33nzE5MCNsZxCGEKTXaQ.png,Marvel Studios,US,299534
1,"444,574,25,290",/42UPdZl6B2cFXgNUASR8hSt9mpS.png,Dune Entertainment|Lightstorm Entertainment|20...,US|US|US|GB,19995
2,"1634,1,11461",None,Truenorth Productions|Lucasfilm|Bad Robot,IS|US|US,140607
3,420,/hUzeosd33nzE5MCNsZxCGEKTXaQ.png,Marvel Studios,US,299536
4,"4,574,25",/fycMZt242LVjagMByZOLUGbCvv3.png,Paramount|Lightstorm Entertainment|20th Centur...,US|US|US,597


3. __Load__ the datasets __into the database__ (each dataset should be a separate table in the database). __Name__ the tables "Movies", "Votes", "Genres", "Prod".

In [330]:
movies.to_sql(name='movies',con = database,if_exists='replace',index=False)

OperationalError: table "movies" already exists

In [ ]:
votes.to_sql(name='votes',con = database,if_exists='replace',index=False)

OperationalError: table "votes" already exists

In [ ]:
genres.to_sql(name='genres',con = database,if_exists='replace',index=False)

18

In [ ]:
prod.to_sql(name='prod',con = database,if_exists='replace',index=False)

18

## Loading Data from SQLite Databases into DataFrames

4. __Load__ the full tables "Movies", "Votes", "Genres", "Prod" from "movies.db" into Pandas (four DataFrames). __Set__ "id" as Index. 

In [ ]:
df = pd.read_sql('SELECT * FROM movies',database)

In [ ]:
df.head()

,id,title,revenue,budget,belongs_to_collection,release_date
0,299534,Avengers: Endgame,2797800564,356000000,The Avengers Collection,2019-04-24 00:00:00
1,19995,Avatar,2787965087,237000000,Avatar Collection,2009-12-10 00:00:00
2,140607,Star Wars: The Force Awakens,2068223624,245000000,Star Wars Collection,2015-12-15 00:00:00
3,299536,Avengers: Infinity War,2046239637,300000000,The Avengers Collection,2018-04-25 00:00:00
4,597,Titanic,1845034188,200000000,,1997-11-18 00:00:00


##  Some Simple SQL Queries

5. __Perform__ the following simple __SQL Queries__ and __store__ the results in DataFrames:

__Load the full "Movies" Table__.

In [ ]:
movies_table = pd.read_sql('SELECT * FROM movies',database)

__Load the columns "id", "revenue" and "release_date" from "Movies".__ 

In [ ]:
cursor = database.cursor()


In [ ]:
cursor.execute('SELECT id ,revenue,release_date FROM movies ')

In [ ]:
rows = cursor.fetchall()

In [ ]:
for row in rows:
    print(row)


(299534, 2797800564, '2019-04-24 00:00:00')
(19995, 2787965087, '2009-12-10 00:00:00')
(140607, 2068223624, '2015-12-15 00:00:00')
(299536, 2046239637, '2018-04-25 00:00:00')
(597, 1845034188, '1997-11-18 00:00:00')
(135397, 1671713208, '2015-06-06 00:00:00')
(420818, 1656943394, '2019-07-12 00:00:00')
(24428, 1519557910, '2012-04-25 00:00:00')
(168259, 1515047671, '2015-04-01 00:00:00')
(99861, 1405403694, '2015-04-22 00:00:00')
(284054, 1346739107, '2018-02-13 00:00:00')
(12445, 1341511219, '2011-07-07 00:00:00')
(181808, 1332539889, '2017-12-13 00:00:00')
(330457, 1330764959, '2019-11-20 00:00:00')
(351286, 1303459585, '2018-06-06 00:00:00')
(109445, 1274219009, '2013-11-27 00:00:00')
(321612, 1263521126, '2017-03-16 00:00:00')
(260513, 1241891456, '2018-06-14 00:00:00')


__Get the Total Revenue (sum) over all movies from "Movies".__

In [ ]:
cursor.execute('SELECT SUM(revenue) FROM Movies')
TRevenue = cursor.fetchall()
print(TRevenue)

[(29748575327,)]


__Count the number of Movies in "Movies".__

In [ ]:
cursor.execute('SELECT COUNT(*) FROM Movies')
No_of_Movies = cursor.fetchall()
print(No_of_Movies)

[(18,)]


__Count the number of Movies that do belong to a collection.__

In [ ]:
cursor.execute("""SELECT COUNT(*) FROM Movies WHERE belongs_to_collection IS NOT NULL AND belongs_to_collection != ''""")
No_of_Franchise_Movies = cursor.fetchall()
print(No_of_Franchise_Movies)

[(15,)]


__Get the average budget from "Movies"__.

In [ ]:
cursor.execute("""SELECT AVG(budget) FROM movies""")
average_budget = cursor.fetchall()
print(average_budget)


[(209055555.55555555,)]


## Some more SQL Queries

6. __Perform__ the following advanced __SQL Queries__ and __store__ the results in DataFrames:

__Load all columns for the movie with movie id 597__.

In [331]:
pd.read_sql("""SELECT * FROM movies WHERE id LIKE '%597%' """,con=database)

,id,title,revenue,budget,belongs_to_collection,release_date
0,597,Titanic,1845034188,200000000,,1997-11-18 00:00:00


__Load all columns for all movies with a revenue greater than 2000 (MUSD).__

In [333]:
pd.read_sql("""SELECT * FROM movies WHERE revenue > 2000000000""",con=database)

,id,title,revenue,budget,belongs_to_collection,release_date
0,299534,Avengers: Endgame,2797800564,356000000,The Avengers Collection,2019-04-24 00:00:00
1,19995,Avatar,2787965087,237000000,Avatar Collection,2009-12-10 00:00:00
2,140607,Star Wars: The Force Awakens,2068223624,245000000,Star Wars Collection,2015-12-15 00:00:00
3,299536,Avengers: Infinity War,2046239637,300000000,The Avengers Collection,2018-04-25 00:00:00


__Load all columns for all movies with a revenue greater than 1500 (MUSD) and a budget below 200 (MUSD).__

In [334]:
pd.read_sql("""SELECT * FROM movies WHERE (revenue > 1500000000 AND budget < 200000000)""",con=database)

,id,title,revenue,budget,belongs_to_collection,release_date
0,135397,Jurassic World,1671713208,150000000,Jurassic Park Collection,2015-06-06 00:00:00
1,168259,Furious 7,1515047671,190000000,The Fast and the Furious Collection,2015-04-01 00:00:00


__Get the minimum budget from those movies with a revenue greater than 1250 (MUSD).__

In [335]:
pd.read_sql("""SELECT MIN(budget) FROM movies WHERE revenue > 1500000000""",con=database)

,MIN(budget)
0,150000000


__Get all unique collection Names from "Movies".__

In [338]:
pd.read_sql("""SELECT distinct(belongs_to_collection) FROM movies""",con=database)

,belongs_to_collection
0,The Avengers Collection
1,Avatar Collection
2,Star Wars Collection
3,
4,Jurassic Park Collection
5,The Fast and the Furious Collection
6,Black Panther Collection
7,Harry Potter Collection
8,Frozen Collection
9,The Incredibles Collection


__Load all movies (all columns) and sort by budget from high to low.__

In [340]:
pd.read_sql("""SELECT * FROM movies ORDER BY budget DESC""",con=database)

,id,title,revenue,budget,belongs_to_collection,release_date
0,299534,Avengers: Endgame,2797800564,356000000,The Avengers Collection,2019-04-24 00:00:00
1,299536,Avengers: Infinity War,2046239637,300000000,The Avengers Collection,2018-04-25 00:00:00
2,420818,The Lion King,1656943394,260000000,,2019-07-12 00:00:00
3,99861,Avengers: Age of Ultron,1405403694,250000000,The Avengers Collection,2015-04-22 00:00:00
4,140607,Star Wars: The Force Awakens,2068223624,245000000,Star Wars Collection,2015-12-15 00:00:00
5,19995,Avatar,2787965087,237000000,Avatar Collection,2009-12-10 00:00:00
6,24428,The Avengers,1519557910,220000000,The Avengers Collection,2012-04-25 00:00:00
7,597,Titanic,1845034188,200000000,,1997-11-18 00:00:00
8,284054,Black Panther,1346739107,200000000,Black Panther Collection,2018-02-13 00:00:00
9,181808,Star Wars: The Last Jedi,1332539889,200000000,Star Wars Collection,2017-12-13 00:00:00


__Load all movies (all columns) that do not belong to a collection.__

In [343]:
pd.read_sql("""SELECT * FROM Movies WHERE belongs_to_collection == ''""",con=database)

,id,title,revenue,budget,belongs_to_collection,release_date
0,597,Titanic,1845034188,200000000,,1997-11-18 00:00:00
1,420818,The Lion King,1656943394,260000000,,2019-07-12 00:00:00
2,321612,Beauty and the Beast,1263521126,160000000,,2017-03-16 00:00:00


__Load all movies (all columns) that belong to a collection.__

In [347]:
pd.read_sql("""SELECT * FROM movies WHERE belongs_to_collection !='' """,con= database)

,id,title,revenue,budget,belongs_to_collection,release_date
0,299534,Avengers: Endgame,2797800564,356000000,The Avengers Collection,2019-04-24 00:00:00
1,19995,Avatar,2787965087,237000000,Avatar Collection,2009-12-10 00:00:00
2,140607,Star Wars: The Force Awakens,2068223624,245000000,Star Wars Collection,2015-12-15 00:00:00
3,299536,Avengers: Infinity War,2046239637,300000000,The Avengers Collection,2018-04-25 00:00:00
4,135397,Jurassic World,1671713208,150000000,Jurassic Park Collection,2015-06-06 00:00:00
5,24428,The Avengers,1519557910,220000000,The Avengers Collection,2012-04-25 00:00:00
6,168259,Furious 7,1515047671,190000000,The Fast and the Furious Collection,2015-04-01 00:00:00
7,99861,Avengers: Age of Ultron,1405403694,250000000,The Avengers Collection,2015-04-22 00:00:00
8,284054,Black Panther,1346739107,200000000,Black Panther Collection,2018-02-13 00:00:00
9,12445,Harry Potter and the Deathly Hallows: Part 2,1341511219,125000000,Harry Potter Collection,2011-07-07 00:00:00


__Load all movies (all columns) where "Avengers..." is in the title__.

In [349]:
pd.read_sql(""" SELECT * FROM movies WHERE title LIKE 'Avengers%'""",con=database)

,id,title,revenue,budget,belongs_to_collection,release_date
0,299534,Avengers: Endgame,2797800564,356000000,The Avengers Collection,2019-04-24 00:00:00
1,299536,Avengers: Infinity War,2046239637,300000000,The Avengers Collection,2018-04-25 00:00:00
2,99861,Avengers: Age of Ultron,1405403694,250000000,The Avengers Collection,2015-04-22 00:00:00


## Join Queries

7. __Perform__ the following __SQL Join Queries__ and __store__ the results in DataFrames:

__Join "Movies" and "Votes"__ (all columns).

In [360]:
pd.read_sql(""" SELECT * FROM movies INNER JOIN Votes ON Movies.id = Votes.id""",con=database)

,id,title,revenue,budget,belongs_to_collection,release_date,id,vote_count,vote_average
0,299534,Avengers: Endgame,2797800564,356000000,The Avengers Collection,2019-04-24 00:00:00,299534,12607,8.3
1,19995,Avatar,2787965087,237000000,Avatar Collection,2009-12-10 00:00:00,19995,21000,7.4
2,140607,Star Wars: The Force Awakens,2068223624,245000000,Star Wars Collection,2015-12-15 00:00:00,140607,14205,7.4
3,299536,Avengers: Infinity War,2046239637,300000000,The Avengers Collection,2018-04-25 00:00:00,299536,17718,8.3
4,597,Titanic,1845034188,200000000,,1997-11-18 00:00:00,597,16661,7.8
5,135397,Jurassic World,1671713208,150000000,Jurassic Park Collection,2015-06-06 00:00:00,135397,15399,6.6
6,420818,The Lion King,1656943394,260000000,,2019-07-12 00:00:00,420818,5425,7.2
7,24428,The Avengers,1519557910,220000000,The Avengers Collection,2012-04-25 00:00:00,24428,22101,7.7
8,168259,Furious 7,1515047671,190000000,The Fast and the Furious Collection,2015-04-01 00:00:00,168259,7359,7.3
9,99861,Avengers: Age of Ultron,1405403694,250000000,The Avengers Collection,2015-04-22 00:00:00,99861,15548,7.3


__Join "Movies" and "Votes" (only the columns "id", "title", "vote_average").__

In [361]:
pd.read_sql("""SELECT Movies.id ,movies.title , Votes.vote_average FROM Movies INNER JOIN Votes ON movies.id = Votes.id""",con=database)

,id,title,vote_average
0,299534,Avengers: Endgame,8.3
1,19995,Avatar,7.4
2,140607,Star Wars: The Force Awakens,7.4
3,299536,Avengers: Infinity War,8.3
4,597,Titanic,7.8
5,135397,Jurassic World,6.6
6,420818,The Lion King,7.2
7,24428,The Avengers,7.7
8,168259,Furious 7,7.3
9,99861,Avengers: Age of Ultron,7.3


__Join "Movies" and "Votes" (only the columns "id", "title", "vote_average") and return only those movies with vote_average > 8.__

In [ ]:
pd.read_sql("""SELECT Movies.id ,movies.title , Votes.vote_average FROM Movies INNER JOIN Votes ON movies.id = Votes.id WHERE Votes.vote_average > 8""",con=database)

,id,title,vote_average
0,299534,Avengers: Endgame,8.3
1,299536,Avengers: Infinity War,8.3
2,12445,Harry Potter and the Deathly Hallows: Part 2,8.1


__Join "Movies" and "Votes" (only the columns "id", "title", "vote_average") and return only those movies with vote_average > 8 and in ascending budget order__.

In [364]:
pd.read_sql("""SELECT Movies.id ,movies.title , Votes.vote_average FROM Movies INNER JOIN Votes ON movies.id = Votes.id WHERE Votes.vote_average > 8 ORDER BY vote_average """,con=database)

,id,title,vote_average
0,12445,Harry Potter and the Deathly Hallows: Part 2,8.1
1,299534,Avengers: Endgame,8.3
2,299536,Avengers: Infinity War,8.3


## Final Case Study

8. __Perform__ the following advanced __SQL Queries__ and __store__ the results in DataFrames:

__Get the Total Revenue (sum) for each Production Company.__

In [367]:
pd.read_sql(""" SELECT movies.revenue,prod.comp_name FROM Movies INNER JOIN prod ON movies.id = prod.id GROUP BY comp_name """,con=database)

,revenue,comp_name
0,1515047671,Abu Dhabi Film Commission|Universal Pictures|C...
1,1303459585,Amblin Entertainment|Legendary Entertainment|U...
2,2787965087,Dune Entertainment|Lightstorm Entertainment|20...
3,1671713208,Fuji Television Network|Amblin Entertainment|L...
4,1332539889,Lucasfilm|Ram Bergman Productions|Walt Disney ...
5,2797800564,Marvel Studios
6,1519557910,Marvel Studios|Paramount
7,1346739107,Marvel Studios|Walt Disney Pictures
8,1845034188,Paramount|Lightstorm Entertainment|20th Centur...
9,2068223624,Truenorth Productions|Lucasfilm|Bad Robot


__Get all Production Companies for the movie "Titanic".__

In [376]:
pd.read_sql(""" SELECT * FROM movies WHERE title = 'Titanic' """,con=database)

,id,title,revenue,budget,belongs_to_collection,release_date
0,597,Titanic,1845034188,200000000,,1997-11-18 00:00:00


In [379]:
pd.read_sql(""" SELECT prod.comp_name FROM movies INNER JOIN prod ON prod.id = movies.id WHERE Movies.title = 'Titanic' """,con=database)

,comp_name
0,Paramount|Lightstorm Entertainment|20th Centur...


__Get the Total Revenue (sum) for each Genre.__

In [385]:
pd.read_sql(""" SELECT movies.revenue,genres.genre_name FROM Movies INNER JOIN genres ON movies.id = genres.id GROUP BY genre_name """,con=database)

,revenue,genre_name
0,1241891456,Action|Adventure|Animation|Family
1,2787965087,Action|Adventure|Fantasy|Science Fiction
2,1405403694,Action|Adventure|Science Fiction
3,2068223624,Action|Adventure|Science Fiction|Fantasy
4,1671713208,Action|Adventure|Science Fiction|Thriller
5,1515047671,Action|Thriller
6,2046239637,Adventure|Action|Science Fiction
7,1656943394,Adventure|Family
8,2797800564,Adventure|Science Fiction|Action
9,1274219009,Animation|Adventure|Family


__Get all Genres for the movie "Frozen II".__

# +++++++++ See some Hints below +++++++++++++

# ++++++++++++++++ Hints++++++++++++++++++++

__Hints for 1.__<br>
You can do this with sq3.connect("database_name.db")

__Hints for 2.__ <br>
You have to use pd.json_normalize(data = ..., record_path = ..., meta = ..., record_prefix = ... ) for Datasets #3 and #4 

__Hints for 3.__<br>
You can do this with: 

In [ ]:
con = sq3.connect("movies.db")
df.to_sql("Table Name", con, index = False)

__Hints for 4.__<br>
You can do this with:

In [ ]:
con = sq3.connect("movies.db")
pd.read_sql("SELECT * FROM Table Name", con, index_col = ...)

__Hints for 5., 6., 7., 8.__<br>
You can do this with:

In [ ]:
con = sq3.connect("movies.db")
df = pd.read_sql("insert the sql query here", con)